# <u>Gradient Boosting Regressor</u>

### Prerequisites:

* <a href="../7.Descision Trees Regression/Descision Trees Regression.ipynb">Check out the notebookes on Decision Tree Regression</a>
* <a href="../8.Random Forest Regression/Random Forest Regression.ipynb">Check out the notebooks on Random Forest Regression</a>

## Topics

* [1. What is Boosting?](#what)
* [2. Boosting vs. Bagging](#vs)
* [3. Forward Stagewise Additive Modeling](#forward)
* [4. Gradient Boosting](#gradient)
* [5. Regularization in Gradient Boosting](#reg)
* [6. Stochastic Gradient Boosting](#stochastic)
* [7. Gradient Boosting for Classification](#class)
* [8. Multiclass Gradient Boosting](#multi)
* [9. Gradient Boosting with Trees](#trees)
* [10. XGBoost (Extreme Gradient Boosting)](#xg)
* [11. XGBoost Regularization](#xg_reg)
* [12. Important XGBoost Hyperparameters](#xg_hyper)
* [13. Key Takeaways](#takeaway)

<a href="../../2.Classification/10.Gradient Boosting Classifier/Gradient Boosting Classification.ipynb">Check out the notebook on Gradient Boosting Classification for more code</a>

In [ ]:
import numpy as np # for arrays and random numbers
import matplotlib.pyplot as plt # for plotting
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting

from sklearn.tree import (
    DecisionTreeClassifier, # for Classification Trees
    DecisionTreeRegressor # for Regression Trees
)


from sklearn.ensemble import (
    AdaBoostClassifier, # for AdaBoost in Classification
    AdaBoostRegressor, # for Adaboost in Regression
    GradientBoostingClassifier, # Gradient Boosting Classification
    GradientBoostingRegressor, # Gradient Boosting Regression
    HistGradientBoostingClassifier, # Histogram-based Gradient Boosting for Classification
    HistGradientBoostingRegressor, # Histogram-based Gradient Boosting for Regression
    RandomForestClassifier, # for Random Forest Classification
    RandomForestRegressor # for Random Forest Regression
)

# XGBoost
#from xgboost import XGBClassifier, XGBRegressor

# LightGBM
#from lightgbm import LGBMClassifier, LGBMRegressor

# CatBoost
#from catboost import CatBoostClassifier, CatBoostRegressor

from sklearn.datasets import make_classification # toy data for classification
from sklearn.datasets import make_regression # toy data for regression

from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score

print("Setup complete")

Setup complete


<a class="anchor" id="what"></a>
# 1. What is Boosting?

- Boosting is an ensemble learning technique that combines many weak learners into a strong predictive model
- **Idea:** <u>Sequentially</u> apply a weak learner to <u>modified versions of the training data</u> with more focus on difficult observations
- Typical weak learners are decision trees and tree stumps (1-level decision trees)
- Boosting can be used fpr Classification, Regression and multiclass porblemns
- Weak learners are a prediction rule with a correct classification rate only slightly better than random guessing (so above 50% accuracy)

<a class="anchor" id="vs"></a>
# 2. Boosting vs. Bagging


<div style="display:flex; gap:20px;">

<div style="
padding:16px;
border-radius:8px;
width:50%;
">

### Bagging / Random Forests
- Base learners are decision trees
- Base learners are trained independently
- Equal weighting for base learners
- Goal: Mainly reduces variance
- Usually resistant to overfitting



</div>

<div style="
padding:16px;
border-radius:8px;
width:50%;
">

### Boosting
- Base learners are (most commonly) weak decision trees 
- Base learners are trained sequentially
- Following base learners focus on errors of previous base learners
- Different weighting for different base learners
- Goal: Reduces both bias and variance
- More prone to overfitting

</div>
</div>

<p align="center">
<img src="pics/1.png" width="500"/>
</p>


<a class="anchor" id="forward"></a>
# 3. Forward Stagewise Additive Modeling

**Gradient boosting builds a model additively:**

$$
f(x) = \sum_{m=1}^M \alpha^{[m]} b(x,\theta^{[m]})
$$

- $b(x,\theta^{[m]})$ is the $m$-th base learner
- $\alpha^{[m]}$ is weight of the $m$-th learner

Goal: Minimize empirical risk $\mathcal{R}_\text{emp}(f)=\sum_{i=1}^n L(y^{(i)},f(x^{(i)}))=\sum_{i=1}^n L(y^{(i)},\sum_{m=1}^M \alpha^{[m]} b(x^{(i)},\theta^{[m]}))$

**Instead of optimizing all learners at once, boosting:**
1. Starts with an initial model $\hat{f}^{[0]}(x)$
2. Adds one learner $b(x^{(i)},\theta)$ at a time and scales it with $\alpha \in [0,1]$ 
3. Greedily minimizes loss  at each step with respect to the nwxt additive component

$$
\min_{\alpha, \theta}\sum_{i=1}^n L(y^{(i)},\underbrace{\hat{f}^{[m-1]}(x^{(i)}) + \alpha b(x^{(i)},\theta)}_{\hat{f}^{[m]}})
$$

This process is called **forward stagewise additive modeling**.

Advantages:
- Easier optimization
- Natural regularization
- Enables early stopping


<a class="anchor" id="pseudo"></a>
# 4. Pseudo Residuals

$$
\tilde{r}^{(i)}=-\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}
$$

For squarred error loss (L2-loss):

$$
\tilde{r}^{(i)}=-\frac{\partial 0.5(y-f(x))}{\partial f(x)}=y-f(x)
$$

### Boosting as Gradient Descent

**We are at the model $f^{[m-1]}$ during minimization and at this point calculate the direction of the negative gradient also called the pseudo-residuals:**

$$
\hat{r}^{[m](i)}=-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f=f^{[m-1]}}
$$

The gradient descent update is:

$$
f^{[m]}(x^{(i)})=f^{[m-1]}(x^{(i)})-\alpha \frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f^{[m-1]}(x^{(i)})}
$$

- $\alpha \frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f^{[m-1]}(x^{(i)})}$ is movement of function $f$ in direction of data to reduce empirical risk




<a class="anchor" id="gradient_reg"></a>
# 4. Gradient Boosting for Regression

<p align="center">
<img src="pics/2.png" width="500"/>
</p>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/3.png" width="550"/>
  <img src="pics/4.png" width="550"/>
</div>


Gradient Boosting interprets boosting as gradient descent in function space.

## Main Idea

At each iteration:
1. Compute pseudo-residuals $\hat{r}^{[m](i)}$ (negative gradients of loss)
2. Fit a weak learner $b(x,\theta^{[m]})$ to these pseudo-residuals
$$
\hat{\theta}^{[m]}=\argmin_\theta \sum_{i=1}^n(\hat{r}^{[m](i)}-b(x^{(i)},\theta))^2
$$
3. Add learner to ensemble

For squared error loss:

$$
\hat{r}^{[m](i)}=y^{(i)}-\hat{f}^{[m-1]}(x^{(i)})
$$

These pseudo-residuals are ordinary residuals.

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/5.png" width="550"/>
  <img src="pics/6.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/7.png" width="550"/>
</div>

### Gradient Boosting Algorithm

1. Initialize $\hat{f}^{[0]}(x)=\underset{\theta_0 \in \mathbb{R}}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta_0)$
2. for $m=1 \rightarrow M$ do
    - For all $i$: $\tilde{r}^{[m](i)}=\underbrace{-\left[\frac{\partial L(y,f)}{\partial f}\right]_{f=\hat{f}^{[m-1]}(x^{(i)}),y=y^{(i)}}}_{\text{Gradient}}$
    - Fit a regression base learner to the vector of pseudo-residuals $\tilde{r}^{[m]}$: $\hat{\theta}^{m}=\underset{\theta}{\argmin}\sum_{i=1}^n (\tilde{r}^{[m](i)}-b(x^{(i)},\theta))^2$
    - Set $\alpha^{[m]}$ to $\alpha$ or via line search
    - Update $\hat{f}^{[m]}=\hat{f}^{[m-1]}(x^{(i)}) + \alpha^{[m]} b(x,\hat{\theta}^{m})$
3. end for
4. Output: $\hat{f}(x)=\hat{f}^{[M]}(x)$


##### Line Search
- Learning rate $\alpha^{[m]}$ controls update size
- Commonly a learning rate of 0.1 is used for all base learners
- However we can also replace it by a line search

**Line Search is an iterative appoach to find a local minimum. So solve**

$$
\hat{\alpha}^{[m]}=\underset{\alpha}{\argmin} \sum_{i=1}^n L(y^{(i)},\hat{f}^{[m-1]}(x^{(i)}) + \alpha b(x,\hat{\theta}^{m}))
$$

### Gradient Boosting for Regression (Simple Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Height (m)} & \textbf{Favorite Color} & \textbf{Gender} & \textbf{Weight (kg)} \\
\hline
1.6 & \text{Blue}  & \text{Male}   & 88 \\
\hline
1.6 & \text{Green} & \text{Female} & 76 \\
\hline
1.5 & \text{Blue}  & \text{Female} & 56 \\
\hline
1.8 & \text{Red}   & \text{Male}   & 73 \\
\hline
1.5 & \text{Green} & \text{Male}   & 77 \\
\hline
1.4 & \text{Blue}  & \text{Female} & 57 \\
\hline
\end{array}
$$

##### Goal: Predict continuous value "Weight".

##### Process: Fit a model to this training data with Gradient Boosting for Regression

##### Base Learner: Decision Trees for Regression

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/8.jpg" width="550"/>
  <img src="pics/9.jpg" width="550"/>
</div>

---
<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/10.jpg" width="550"/>
  <img src="pics/11.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/12.jpg" width="550"/>
  <img src="pics/13.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/15.jpg" width="550"/>
  <img src="pics/16.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/17.jpg" width="550"/>
  <img src="pics/18.jpg" width="550"/>
</div>


### Gradient Boosting for Regression (Detailed Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Height (m)} & \textbf{Favorite Color} & \textbf{Gender} & \textbf{Weight (kg)} \\
\hline
1.6 & \text{Blue}  & \text{Male}   & 88 \\
\hline
1.6 & \text{Green} & \text{Female} & 76 \\
\hline
1.5 & \text{Blue}  & \text{Female} & 56 \\
\hline
\end{array}
$$


##### Goal: Predict continuous value Weight.

##### Process: Fit a model to this training data with Gradient Boosting for Regression

##### Base Learner: Decision Trees for Regression

**Note: Since we only have 3 samples the resulting trees in Gradient Boosting with Regression Trees will only have two leaves.**

### Gradient Boosting Algorithm with Regression Trees as base learners

Input: Data $\left\{(x^{(i)},y^{(i)})\right\}_{i=1}^n$ and a differentiable Loss function $L(y,f(x))$
1. Initialize model with a constant value: $f_0(x)=\underset{\theta}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta)$
2. for $m=1$ to $M$ do
    - (A) Compute pseudo-residuals $r_{im}=\underbrace{-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f(x^{(i)})=f_{m-1}(x^{(i)})}}_{\text{Gradient}}$ for $i=1,\ldots,n$
    - (B) Fit a regression tree to the $r^{im}$ pseudo-residuals and create terminal regions $R_{jm}$ for $j=1,\ldots,J_m$ 

    - (C) For $j=1,\ldots,J_m$ compute $\theta_{jm}=\underset{\theta}{\argmin}\underset{x^{(i)} \in R_{jm}}{\sum} L(y^{(i)},f_{m-1}(x^{(i)})+\theta)$
    - (D) Update $f_m=f_{m-1}(x) + \alpha \sum_{j=1}^{J_m} \theta_{jm} \mathbb{I}(x \in R_{jm}) $
3. end for
4. Output: $f_{M}(x)$



<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/19.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/20.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/21.jpg" width="600"/>
</div>

---


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/22.jpg" width="600"/>
  <img src="pics/23.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/24.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/25.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/26.jpg" width="600"/>
  <img src="pics/27.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/28.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/29.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/30.jpg" width="600"/>
</div>

In [ ]:
# Gradient Boost Regression with Regression Trees

### Gradient Boosting for Classification (Simple Example)

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Likes Popcorn} & \textbf{Age} & \textbf{Favorite Color} & \textbf{Loves movies} \\
\hline
\text{Yes} & 12  & \text{Blue}   & \text{Yes} \\
\hline
\text{Yes} & 87 & \text{Green} & \text{Yes} \\
\hline
\text{No} & 44  & \text{Blue} & \text{No} \\
\hline
\text{Yes} & 19  & \text{Red}   & \text{No} \\
\hline
\text{No} & 32 & \text{Green}   & \text{Yes} \\
\hline
\text{No} & 14  & \text{Blue} & \text{Yes} \\
\hline
\end{array}
$$

##### Goal: Predict categorical/binary target "Loves Movies" 

##### Process: Fit a model to this training data with Gradient Boosting for Classification

##### Base Learner: Decision Trees for Classification

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/31.jpg" width="550"/>
  <img src="pics/32.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/33.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/34.jpg" width="550"/>
  <img src="pics/35.jpg" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/36.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/37.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/38.jpg" width="600"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/39.jpg" width="600"/>
</div>



### Gradient Boosting Algorithm with Classification Trees as base learners

Input: Data $\left\{(x^{(i)},y^{(i)})\right\}_{i=1}^n$ and a differentiable Loss function $L(y,f(x))$
1. Initialize model with a constant value: $f_0(x)=\underset{\theta}{\argmin}\sum_{i=1}^n L(y^{(i)},\theta)$
2. for $m=1$ to $M$ do
    - (A) Compute pseudo-residuals $r_{im}=\underbrace{-\left[\frac{\partial L(y^{(i)},f(x^{(i)}))}{\partial f(x^{(i)})}\right]_{f(x^{(i)})=f_{m-1}(x^{(i)})}}_{\text{Gradient}}$ for $i=1,\ldots,n$
    - (B) Fit a regression tree to the $r^{im}$ pseudo-residuals and create terminal regions $R_{jm}$ for $j=1,\ldots,J_m$ 

    - (C) For $j=1,\ldots,J_m$ compute $\theta_{jm}=\underset{\theta}{\argmin}\underset{x^{(i)} \in R_{jm}}{\sum} L(y^{(i)},f_{m-1}(x^{(i)})+\theta)$
    - (D) Update $f_m=f_{m-1}(x) + \alpha \sum_{j=1}^{J_m} \theta_{jm} \mathbb{I}(x \in R_{jm}) $
3. end for
4. Output: $f_{M}(x)$

<a class="anchor" id="reg"></a>
# 5. Regularization in Gradient Boosting

Gradient Boosting can easily overfit, so regularization is important.

Main regularization methods:

1. Number of Iterations $M$
- Fewer base learners (additive components) reduce overfitting (Early stopping)

2. Tree Depth
- When using decision trees as base learners limit their maximum depth 
- Limiting tree depth also controls interaction complexity
- Small depth $\rightarrow$ simpler models

3. Learning Rate (Shrinkage)
- Small learning rate $\alpha$ slows learning
- Usually improves generalization
- Requires more base learners though

**Typical tradeoff: Small learning rate & choose $M$ via cross validation.**

<a class="anchor" id="stochastic"></a>
# 6. Stochastic Gradient Boosting

Adds randomness to gradient boosting.

At each iteration:

- Train on a random subset of data

Benefits:

- Reduces overfitting
- Improves generalization
- Similar idea to bagging

Additional hyperparameter: Subsample ratio (size of random data sets)

<a class="anchor" id="class"></a>
# 7. Gradient Boosting for Classification

<a class="anchor" id="class"></a>
## 7.1 Gradient Boosting for Binary Classification

For binary classification, Gradient Boosting uses classification loss functions.

Most common:
- Bernoulli / Logistic loss

Pseudo-residuals become:

$$
\hat{r}^{[m](i)}= y^{(i)}-\pi(x^{(i)})
$$

where:
- $\pi(x^{(i)})$ = predicted probability

Important notes:
- Regression trees are still fitted to residuals
- Predicted probabilities come from logistic transformation

Using exponential loss makes Gradient Boosting closely related to AdaBoost.


<a class="anchor" id="class"></a>
## 7.2 Gradient Boosting for Multiclass Classification

<a class="anchor" id="trees"></a>
# 9. Gradient Boosting with Trees

Decision trees are the most common base learners.

Advantages:
- Handle categorical data
- Robust to outliers
- Handle missing values
- Fast training
- Automatic variable selection
- Interaction Depth

Tree depth determines interaction complexity.

- Depth 1 (stumps):
    - additive model only
    - no feature interactions
- Depth 2+:
    - captures feature interactions

Gradient Boosting with trees is highly flexible and powerful.


<a class="anchor" id="xg"></a>
# 10. XGBoost (Extreme Gradient Boosting)

XGBoost is an optimized implementation of Gradient Boosting.

Main features:
- Fast and scalable
- Parallelized tree construction
- GPU support
- Regularization
- Subsampling
- Efficient split finding

Widely used in:
- Structured/tabular data tasks
- Industry ML systems

<a class="anchor" id="xg_reg"></a>
# 11. XGBoost Regularization

XGBoost introduces additional regularization terms.

Penalties
1. Tree complexity penalty
    - penalizes too many leaves
2. L2 regularization
    - shrinks leaf weights
3. L1 regularization
    - encourages sparsity

Benefits:
- Better generalization
- Reduced overfitting
- More stable trees

XGBoost also uses:
- feature subsampling
- data subsampling
- pruning
- dropout-style boosting (DART)


<a class="anchor" id="xg_hyper"></a>
# 12. Important XGBoost Hyperparameters

| Hyperparameter     | Purpose                          |
| ------------------ | -------------------------------- |
| `eta`              | Learning rate                    |
| `nrounds`          | Number of boosting iterations    |
| `max_depth`        | Maximum tree depth               |
| `gamma`            | Minimum loss reduction for split |
| `subsample`        | Fraction of training rows        |
| `colsample_bytree` | Fraction of features per tree    |
| `lambda`           | L2 regularization                |
| `alpha`            | L1 regularization                |


<a class="anchor" id="takeaway"></a>
# 13. Key Takeaways